# 04 — ML training
Builds next-interval kWh features and trains a candidate model.

In [ ]:
import os
import re
from datetime import datetime, timezone
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.sql import Window, functions as F

catalog = os.getenv('AIDP_CATALOG', 'aidp_poc')
if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', catalog):
    raise ValueError('AIDP_CATALOG must be a simple Spark identifier')
silver_table = f'{catalog}.silver.interval_reading'
features_table = f'{catalog}.ml.meter_features'
base = spark.table(silver_table).where("quality_code = 'ACTUAL'")

window = Window.partitionBy('meter_id').orderBy('interval_start_utc')
features = (base.select('meter_id', 'interval_start_utc', 'reading_date', 'consumption_kwh', 'temperature_c', 'voltage_v', 'tariff_code')
    .withColumn('target_next_kwh', F.lead('consumption_kwh', 1).over(window))
    .withColumn('lag_1_kwh', F.lag('consumption_kwh', 1).over(window))
    .withColumn('lag_4_kwh', F.lag('consumption_kwh', 4).over(window))
    .withColumn('lag_96_kwh', F.lag('consumption_kwh', 96).over(window))
    .withColumn('rolling_4_kwh', F.avg('consumption_kwh').over(window.rowsBetween(-4, -1)))
    .withColumn('hour', F.hour('interval_start_utc'))
    .withColumn('day_of_week', F.dayofweek('interval_start_utc'))
    .withColumn('is_weekend', F.when(F.dayofweek('interval_start_utc').isin(1, 7), 1).otherwise(0))
    .drop('consumption_kwh').dropna().withColumn('ts_epoch', F.col('interval_start_utc').cast('long')))

if features.limit(1).count() == 0:
    raise ValueError('Insufficient history: each meter needs at least 97 valid 15-minute ACTUAL readings before training.')
features.drop('ts_epoch').write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(features_table)
cutoff = features.approxQuantile('ts_epoch', [0.80], 0.01)[0]
train = features.where(F.col('ts_epoch') < cutoff)
test = features.where(F.col('ts_epoch') >= cutoff)
if train.limit(1).count() == 0 or test.limit(1).count() == 0:
    raise ValueError('Training requires readings spanning more than one time split.')

numeric = ['lag_1_kwh', 'lag_4_kwh', 'lag_96_kwh', 'rolling_4_kwh', 'hour', 'day_of_week', 'is_weekend', 'temperature_c', 'voltage_v']
pipeline = Pipeline(stages=[StringIndexer(inputCol='tariff_code', outputCol='tariff_idx', handleInvalid='keep'), VectorAssembler(inputCols=numeric + ['tariff_idx'], outputCol='features'), GBTRegressor(labelCol='target_next_kwh', maxIter=75, maxDepth=6, seed=42)])
model = pipeline.fit(train)
predictions = model.transform(test)
metrics = {metric: RegressionEvaluator(labelCol='target_next_kwh', predictionCol='prediction', metricName=metric).evaluate(predictions) for metric in ('mae', 'rmse')}
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
model_path = f'/Workspace/models/smart_meter_next_kwh/{run_id}'
model.write().overwrite().save(model_path)
print(f'Saved candidate model to {model_path}; validation metrics={metrics}. Promote it through your governed model registry.')